# 04 — Options Strategy Analyzer

## Why this project exists

A trader often wants a fast answer to: **"What is my exact risk/reward if I put on this option structure?"**

This notebook builds a simple payoff analyzer for common option structures:

- bull put spread
- bear call spread
- iron condor
- 1-2-1 put butterfly

The goal is to demonstrate that I understand option payoff logic and can build practical tools that help frame trading decisions. This is not a pricing model and it does not estimate probability of profit. It is a clear expiry payoff tool.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = "{:,.2f}".format

## 1. Define reusable payoff functions

I keep the function logic simple and transparent. For desk support work, this matters: traders need to trust the output quickly.

In [ ]:
def option_payoff(S, option_type, strike, premium, quantity=1):
    S = np.asarray(S)
    if option_type == "call":
        intrinsic = np.maximum(S - strike, 0)
    elif option_type == "put":
        intrinsic = np.maximum(strike - S, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")
    return quantity * (intrinsic - premium)


def strategy_payoff(S, legs):
    total = np.zeros_like(np.asarray(S), dtype=float)
    for leg in legs:
        total += option_payoff(S, leg["type"], leg["strike"], leg["premium"], leg["quantity"])
    return total

## 2. Create example strategies

These prices are manually entered examples. In a production setting, I would connect this to an option-chain API or broker data source. For a public GitHub portfolio, manually entered examples are safer and more reproducible.

In [ ]:
spot = 500
S = np.linspace(400, 600, 401)

strategies = {
    "Bull Put Spread": [
        {"type": "put", "strike": 490, "premium": 4.00, "quantity": -1},
        {"type": "put", "strike": 470, "premium": 1.75, "quantity": 1},
    ],
    "Bear Call Spread": [
        {"type": "call", "strike": 510, "premium": 4.20, "quantity": -1},
        {"type": "call", "strike": 530, "premium": 1.80, "quantity": 1},
    ],
    "Iron Condor": [
        {"type": "put", "strike": 470, "premium": 2.00, "quantity": 1},
        {"type": "put", "strike": 485, "premium": 4.10, "quantity": -1},
        {"type": "call", "strike": 515, "premium": 4.00, "quantity": -1},
        {"type": "call", "strike": 530, "premium": 1.90, "quantity": 1},
    ],
    "1-2-1 Put Fly": [
        {"type": "put", "strike": 490, "premium": 2.50, "quantity": 1},
        {"type": "put", "strike": 475, "premium": 1.40, "quantity": -2},
        {"type": "put", "strike": 460, "premium": 0.70, "quantity": 1},
    ],
}

## 3. Visualize payoffs

A payoff chart is one of the fastest ways to understand whether the risk profile matches the trade thesis.

In [ ]:
all_payoffs = []
summary_rows = []

for name, legs in strategies.items():
    payoff = strategy_payoff(S, legs)
    all_payoffs.append(pd.DataFrame({"underlying_price": S, "payoff": payoff, "strategy": name}))
    summary_rows.append({
        "strategy": name,
        "max_profit_in_chart_range": payoff.max(),
        "max_loss_in_chart_range": payoff.min(),
        "best_underlying_price": S[payoff.argmax()],
        "worst_underlying_price": S[payoff.argmin()],
    })

payoff_df = pd.concat(all_payoffs, ignore_index=True)
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
px.line(payoff_df, x="underlying_price", y="payoff", color="strategy", title="Options strategy payoff at expiry").show()

## 4. My conclusion

This project demonstrates practical option tooling:

- Define trade legs clearly
- Aggregate payoff across legs
- Compare risk/reward across strategies
- Visualize outcomes in a way a trader can immediately understand

## Next iterations

- Add commissions
- Add contract multiplier
- Add probability estimates using implied volatility
- Connect to broker option-chain data
- Convert to Streamlit